In [2]:
import random
import time
from sentence_transformers import SentenceTransformer, util
import torch

In [1]:
print("Hello world")

Hello world


In [3]:
#  GENERACION DEL DATASET SIMULADO (500 fichas)
def generar_fichas(cantidad=500):
    sexos = ["hombre", "mujer"]
    complexiones = ["delgada", "robusta", "regular", "atlética"]
    pieles = ["blanco", "moreno claro", "moreno oscuro"]
    cabellos = ["negro corto liso", "castaño largo rizado", "rubio corto", "rapado"]
    senas = ["SIN DATO", "tatuaje de colibrí en el pecho", "cicatriz en la ceja derecha", "tatuaje de cruz en el brazo izquierdo", "lunar grande en la mejilla"]
    prendas = ["SIN DATO", "playera roja y pantalón de mezclilla", "vestido azul", "chamarra de cuero negra y jeans", "sudadera gris y tenis blancos"]

    fichas = []
    for _ in range(cantidad - 2): # Dejamos espacio para 2 inyectadas a mano
        texto = f"Persona de sexo {random.choice(sexos)}. Características físicas: COMPLEXION: {random.choice(complexiones)}, COLOR DE PIEL: {random.choice(pieles)}, CABELLO: {random.choice(cabellos)}."
        
        sena = random.choice(senas)
        if sena != "SIN DATO":
            texto += f" Señas particulares: {sena}."
            
        prenda = random.choice(prendas)
        if prenda != "SIN DATO":
            texto += f" Vestimenta: {prenda}."
            
        fichas.append(texto)

    # Inyectamos 2 fichas especificas para asegurar que el modelo las encuentre
    fichas.append("Persona de sexo hombre. Características físicas: COMPLEXION: delgada, COLOR DE PIEL: blanco. Señas particulares: tatuaje de colibrí en el pecho. Vestimenta: playera negra.")
    fichas.append("Persona de sexo mujer. Características físicas: COMPLEXION: robusta. Señas particulares: cicatriz en la ceja derecha. Vestimenta: vestido azul oscuro.")
    
    return fichas

In [4]:
print("Generando 500 fichas de prueba...")
corpus_fichas = generar_fichas(500)

Generando 500 fichas de prueba...


In [5]:
# 2. CARGAR EL MODELO
print("Cargando modelo SBERT (paraphrase-multilingual-MiniLM-L12-v2)...")
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Cargando modelo SBERT (paraphrase-multilingual-MiniLM-L12-v2)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# 3. VECTORIZAR LAS 500 FICHAS (Batch Processing)
print("Vectorizando las 500 fichas (esto puede tomar unos segundos)...")
t0 = time.time()
# convert_to_tensor=True es necesario para usar util.cos_sim de forma rapida con PyTorch
corpus_embeddings = modelo.encode(corpus_fichas, convert_to_tensor=True, show_progress_bar=True)
print(f"¡Vectorización completa en {time.time() - t0:.2f} segundos!\n")

Vectorizando las 500 fichas (esto puede tomar unos segundos)...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

¡Vectorización completa en 8.14 segundos!



In [7]:
# 4. QUERIES DE PRUEBA (Lo que escribiría el usuario)
consultas_usuario = [
    "Hombre con tatuaje de colibrí",
    "Mujer con vestido azul y cicatriz",
    "Chico de tez morena con chamarra de cuero",
    "Persona rubia",
    "Alguien con un tatuaje de cruz en el brazo"
]

In [8]:
for consulta in consultas_usuario:
    print(f"🔍 BÚSQUEDA DEL USUARIO: '{consulta}'")
    
    # Vectorizar el input del usuario
    query_embedding = modelo.encode(consulta, convert_to_tensor=True)
    
    # Calcular SIMILITUD DE COSENO contra los 500 vectores de la base de datos
    # util.cos_sim devuelve una matriz, tomamos la primera fila [0]
    cosine_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    
    # Obtener los 3 resultados con mayor puntaje
    top_results = torch.topk(cosine_scores, k=3)
    
    print("Top 3 resultados encontrados:")
    # top_results[0] son los puntajes (scores), top_results[1] son los índices (posiciones en la lista)
    for score, idx in zip(top_results[0], top_results[1]):
        # Convertimos el score a un porcentaje para que sea más legible
        porcentaje_similitud = score.item() * 100
        print(f"  [{porcentaje_similitud:.1f}%] - {corpus_fichas[idx]}")
    print("-" * 50)

🔍 BÚSQUEDA DEL USUARIO: 'Hombre con tatuaje de colibrí'
Top 3 resultados encontrados:
  [68.5%] - Persona de sexo hombre. Características físicas: COMPLEXION: atlética, COLOR DE PIEL: moreno claro, CABELLO: rapado. Señas particulares: tatuaje de colibrí en el pecho.
  [68.3%] - Persona de sexo hombre. Características físicas: COMPLEXION: atlética, COLOR DE PIEL: moreno oscuro, CABELLO: rapado. Señas particulares: tatuaje de colibrí en el pecho.
  [67.7%] - Persona de sexo hombre. Características físicas: COMPLEXION: atlética, COLOR DE PIEL: blanco, CABELLO: negro corto liso. Señas particulares: tatuaje de colibrí en el pecho.
--------------------------------------------------
🔍 BÚSQUEDA DEL USUARIO: 'Mujer con vestido azul y cicatriz'
Top 3 resultados encontrados:
  [60.7%] - Persona de sexo mujer. Características físicas: COMPLEXION: robusta. Señas particulares: cicatriz en la ceja derecha. Vestimenta: vestido azul oscuro.
  [57.9%] - Persona de sexo mujer. Características físicas: CO